# ML08 · 訓練的藝術

把一個能跑的神經網路，調教成「在沒看過的資料上也表現好」的模型。

## 學習目標
- 理解小批次（mini-batch）與資料載入器（DataLoader）的概念，知道為什麼不一次餵全部資料。
- 分辨優化器（optimizer）SGD 與 Adam 的差異，知道何時選哪一個。
- 會做訓練/驗證切分（train/validation split），並用它偵測過擬合（overfitting）。
- 學會兩種對抗過擬合的工具：丟棄法（dropout）與提早停止（early stopping）。
- 能判讀學習曲線（learning curve），從曲線形狀診斷訓練狀況並決定下一步。

In [ ]:
# 概念：統一匯入與畫圖設定，後面各範例共用；固定亂數種子讓結果可重現
import numpy as np
import matplotlib.pyplot as plt

# 讓 matplotlib 在 notebook 內直接顯示圖
%matplotlib inline

# 設定中文字型，避免圖上中文變成方框（找不到也不影響數值與英文標籤）
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False   # 負號正常顯示

print("環境就緒，numpy 版本:", np.__version__)

## 小批次與資料載入

小批次（mini-batch）是指每次參數更新只看「一小撮」樣本，而不是全部資料。一次完整看過所有資料叫一個世代（epoch）；一個 epoch 內會切成很多小批次依序更新。

為什麼這樣做：更新更頻繁、記憶體更省，而且小批次帶來的梯度雜訊反而有助於跳出不好的點。批次大小（batch size）設大時更新次數少、每步較穩；設小時更新次數多、雜訊較大。資料載入器（DataLoader）就是「自動分批與洗牌（shuffle）」的搬運工。

形狀：一個 epoch 的更新次數約為 樣本數 / batch size（無條件進位）。

In [ ]:
# 概念：手寫一個迷你分批器（mini-batch iterator），每個 epoch 先洗牌再切成固定大小的批次
rng = np.random.default_rng(0)   # 固定種子，方便對照

# 造一批假樣本：每棟建築有 樓高(公尺) 與 面積(平方公尺) 兩個特徵
n_samples = 20
features = rng.uniform(low=[6, 50], high=[60, 240], size=(n_samples, 2))

def iterate_minibatches(data, batch_size, shuffle=True):
    n = len(data)
    indices = np.arange(n)
    if shuffle:
        rng.shuffle(indices)              # 洗牌：打散樣本順序，避免排序影響更新
    for start in range(0, n, batch_size):
        batch_idx = indices[start:start + batch_size]   # 取這一批的列索引
        yield data[batch_idx]             # 用 yield 逐批吐出，省記憶體

# 比較 batch size 設大與設小時，一個 epoch 的更新次數差別
for batch_size in (4, 10):
    batches = list(iterate_minibatches(features, batch_size))
    sizes = [len(b) for b in batches]
    print(f"batch_size={batch_size}: 一個 epoch 更新 {len(batches)} 次，各批大小 {sizes}")

# 注意：樣本數不一定整除 batch size，最後一批會比較小，這是正常現象

## 優化器之爭：SGD 對上 Adam

優化器（optimizer）決定「拿到梯度後怎麼更新參數」。關鍵直覺一句話：更新就是讓參數沿著負梯度 −∇L 的方向移動，學習率（learning rate）控制每一步走多大。

隨機梯度下降（SGD, stochastic gradient descent）像用固定步幅走下坡；加上動量（momentum）會累積慣性、衝得更順。Adam 則會依每個方向的歷史梯度自動調整步幅（自適應學習率，adaptive learning rate），通常收斂更快更穩。學習率仍是最關鍵的旋鈕。

| 優化器 | 步幅 | 特性 | 何時用 |
|---|---|---|---|
| SGD | 固定 | 單純、需手調學習率 | 想精細控制、資料大時 |
| SGD + momentum | 固定 + 慣性 | 走得較順、較快 | 想加速又保留 SGD 行為 |
| Adam | 自適應 | 收斂快、較不挑學習率 | 一開始的安全預設 |

In [ ]:
# 概念：在同一個自造線性迴歸任務上，分別用 SGD 與 Adam 訓練，比較損失下降曲線
rng = np.random.default_rng(1)

# 造一份線性關係假資料：y = 2*x + 雜訊（x 當成標準化後的某個特徵）
x = rng.normal(size=(60, 1))
true_w, true_b = 2.0, 0.5
y = true_w * x + true_b + 0.1 * rng.normal(size=(60, 1))

def loss_and_grad(w, b, x, y):
    pred = w * x + b
    error = pred - y
    mse = np.mean(error ** 2)            # 均方誤差 mean squared error
    grad_w = np.mean(2 * error * x)      # 對 w 的梯度
    grad_b = np.mean(2 * error)          # 對 b 的梯度
    return mse, grad_w, grad_b

def train_sgd(lr=0.1, epochs=40):
    w, b = 0.0, 0.0
    history = []
    for _ in range(epochs):
        mse, gw, gb = loss_and_grad(w, b, x, y)
        w -= lr * gw                     # 沿負梯度方向移動固定步幅
        b -= lr * gb
        history.append(mse)
    return history

def train_adam(lr=0.1, epochs=40, beta1=0.9, beta2=0.999, eps=1e-8):
    w, b = 0.0, 0.0
    mw = mb = vw = vb = 0.0              # 一階(動量)與二階(梯度平方)的移動平均
    history = []
    for t in range(1, epochs + 1):
        mse, gw, gb = loss_and_grad(w, b, x, y)
        mw = beta1 * mw + (1 - beta1) * gw
        mb = beta1 * mb + (1 - beta1) * gb
        vw = beta2 * vw + (1 - beta2) * gw ** 2
        vb = beta2 * vb + (1 - beta2) * gb ** 2
        # 偏差校正 bias correction：補償初期移動平均偏向 0 的問題
        mw_hat, mb_hat = mw / (1 - beta1 ** t), mb / (1 - beta1 ** t)
        vw_hat, vb_hat = vw / (1 - beta2 ** t), vb / (1 - beta2 ** t)
        w -= lr * mw_hat / (np.sqrt(vw_hat) + eps)   # 每個方向各自的自適應步幅
        b -= lr * mb_hat / (np.sqrt(vb_hat) + eps)
        history.append(mse)
    return history

sgd_hist = train_sgd()
adam_hist = train_adam()

plt.figure(figsize=(6, 4))
plt.plot(sgd_hist, label="SGD")
plt.plot(adam_hist, label="Adam")
plt.xlabel("epoch")
plt.ylabel("訓練損失 MSE")
plt.title("SGD 與 Adam 的收斂比較")
plt.legend()
plt.show()

print("SGD 最終損失:", round(sgd_hist[-1], 5))
print("Adam 最終損失:", round(adam_hist[-1], 5))

## 切出驗證集，看見過擬合

模型在訓練集上變好，不等於它真的「學會」了。泛化（generalization）指的是對沒見過的資料也表現好。要看泛化，就得把一部分資料藏起來當驗證集（validation set）。

過擬合（overfitting）是模型把訓練資料的雜訊都背了下來：訓練損失持續下降，但驗證損失反轉上升，兩者落差越拉越大。反過來，欠擬合（underfitting）是模型太弱，連訓練集都學不好，兩條損失都偏高。

形狀：先打亂索引，依比例切出 train/validation 兩份；訓練只用 train，評估看 validation。

In [ ]:
# 概念：刻意把一個高彈性模型訓練到過擬合，觀察訓練損失下降但驗證損失反轉上升
rng = np.random.default_rng(2)

# 造帶雜訊的假資料：容積率 floor area ratio 對地價（單位簡化）
far_raw = np.linspace(1.0, 5.0, 24).reshape(-1, 1)
land_price = 3.0 * far_raw + 1.2 * rng.normal(size=far_raw.shape) + 2.0   # 近似線性 + 較大雜訊

# 注意：高次方會讓數值爆掉，先把容積率標準化到 [-1, 1] 附近，多項式特徵才不會 overflow
far = (far_raw - far_raw.mean()) / (far_raw.max() - far_raw.min())

# 打亂後切成 train / validation（資料少，故意讓訓練集很小以放大過擬合）
idx = rng.permutation(len(far))
split = 8
train_idx, val_idx = idx[:split], idx[split:]

# 技巧：用高次多項式特徵製造「過度彈性」的模型，最容易示範過擬合
def make_features(v, degree=9):
    return np.hstack([v ** d for d in range(degree + 1)])   # 1, v, v^2, ... 當作多個特徵

X_train = make_features(far[train_idx])
X_val = make_features(far[val_idx])
y_train, y_val = land_price[train_idx], land_price[val_idx]

w = np.zeros((X_train.shape[1], 1))   # 每個多項式特徵一個權重
lr, epochs = 0.5, 60000
train_curve, val_curve = [], []
for _ in range(epochs):
    error = X_train @ w - y_train
    grad = X_train.T @ error / len(X_train)   # 對權重的梯度
    w -= lr * grad
    train_curve.append(np.mean((X_train @ w - y_train) ** 2))
    val_curve.append(np.mean((X_val @ w - y_val) ** 2))   # 驗證損失：沒參與更新

plt.figure(figsize=(6, 4))
plt.plot(train_curve, label="訓練損失")
plt.plot(val_curve, label="驗證損失")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.title("過擬合：訓練損失持續下降、驗證損失反轉")
plt.legend()
plt.show()

print("訓練損失最終:", round(train_curve[-1], 4))
print("驗證損失最低出現在 epoch:", int(np.argmin(val_curve)), "  最終驗證損失:", round(val_curve[-1], 4))

## 正則化工具一：丟棄法 dropout

正則化（regularization）泛指「故意給模型一點限制」以換取更好的泛化。丟棄法（dropout）是其中最常用的一種。

做法：訓練時每一步隨機關掉一部分神經元（依丟棄率 dropout rate），逼網路不依賴單一路徑、學到更穩健的特徵。直覺上像考試前隨機抽掉幾頁筆記練習，反而更會舉一反三。關鍵注意點是訓練與推論模式差異（train/eval mode）：推論（預測）時要關閉 dropout 並做縮放補償，否則輸出尺度會不對。

形狀：訓練時 輸出 = 輸入 * 隨機遮罩 /(1 − rate)；推論時 輸出 = 輸入（不丟棄）。

In [ ]:
# 概念：手寫 inverted dropout，示範訓練模式會丟棄並縮放、推論模式原樣輸出
rng = np.random.default_rng(3)

# 造一層神經元的假輸出（一批 5 個樣本、每樣本 6 個神經元）
activations = rng.uniform(1.0, 2.0, size=(5, 6))

def dropout(x, rate, training):
    if not training:
        return x                       # 注意：推論模式不丟棄，直接回傳
    keep_prob = 1.0 - rate
    mask = (rng.uniform(size=x.shape) < keep_prob)   # 隨機決定哪些神經元保留
    # 除以 keep_prob 做縮放，讓訓練與推論的期望輸出一致
    return x * mask / keep_prob

train_out = dropout(activations, rate=0.5, training=True)
eval_out = dropout(activations, rate=0.5, training=False)

print("訓練模式輸出（部分被歸零、其餘放大）:\n", np.round(train_out, 2))
print("\n推論模式輸出（與原輸入相同）:\n", np.round(eval_out, 2))
# 期望值應接近：縮放讓兩種模式的整體平均對得起來
print("\n原輸入平均:", round(activations.mean(), 3), " 訓練輸出平均:", round(train_out.mean(), 3))

In [ ]:
# 概念：在前面過擬合的多項式模型上，用權重衰減模擬正則化效果，比較驗證損失的改善
rng = np.random.default_rng(2)   # 沿用同一份資料設定，重現過擬合場景

far_raw = np.linspace(1.0, 5.0, 24).reshape(-1, 1)
land_price = 3.0 * far_raw + 1.2 * rng.normal(size=far_raw.shape) + 2.0
far = (far_raw - far_raw.mean()) / (far_raw.max() - far_raw.min())   # 同樣標準化避免 overflow
idx = rng.permutation(len(far))
split = 8
train_idx, val_idx = idx[:split], idx[split:]

def make_features(v, degree=9):
    return np.hstack([v ** d for d in range(degree + 1)])

X_train, X_val = make_features(far[train_idx]), make_features(far[val_idx])
y_train, y_val = land_price[train_idx], land_price[val_idx]

def train(reg_strength):
    w = np.zeros((X_train.shape[1], 1))
    val_curve = []
    for _ in range(60000):
        error = X_train @ w - y_train
        # 正則化項：對權重大小施加懲罰，逼模型別把雜訊背得太用力
        grad = X_train.T @ error / len(X_train) + reg_strength * w
        w -= 0.5 * grad
        val_curve.append(np.mean((X_val @ w - y_val) ** 2))
    return val_curve

no_reg = train(reg_strength=0.0)      # 不正則化（容易過擬合）
with_reg = train(reg_strength=0.005)  # 加入適度正則化
# 技巧：正則化強度要適中；太強反而會欠擬合，可掃幾個值挑驗證損失最低的

plt.figure(figsize=(6, 4))
plt.plot(no_reg, label="未加正則化")
plt.plot(with_reg, label="加入正則化")
plt.xlabel("epoch")
plt.ylabel("驗證損失 MSE")
plt.title("正則化讓驗證損失反轉幅度變小")
plt.legend()
plt.show()

print("未加正則化 最終驗證損失:", round(no_reg[-1], 4), " 反轉後最高:", round(max(no_reg[int(np.argmin(no_reg)):]), 4))
print("加入正則化 最終驗證損失:", round(with_reg[-1], 4), " 反轉後最高:", round(max(with_reg[int(np.argmin(with_reg)):]), 4))

## 正則化工具二：提早停止 early stopping

提早停止（early stopping）是說：與其硬訓練固定世代數，不如盯著驗證損失，一旦它連續好幾個 epoch 不再進步就停手，並保留表現最好的那一版。

三個關鍵詞：監控指標（monitor metric，通常是驗證損失）、耐心值（patience，容忍幾個 epoch 沒進步）、最佳檢查點（best checkpoint，把驗證損失最低時的參數存起來，最後還原）。這樣既省時間又自動避開過擬合區段。

形狀：若驗證損失創新低就更新檢查點並歸零計數；否則計數加一，計數超過 patience 就停。

In [ ]:
# 概念：自寫 early stopping 監控迴圈，記錄並還原驗證損失最低時的參數，標出停在第幾個 epoch
rng = np.random.default_rng(2)

far_raw = np.linspace(1.0, 5.0, 24).reshape(-1, 1)
land_price = 3.0 * far_raw + 1.2 * rng.normal(size=far_raw.shape) + 2.0
far = (far_raw - far_raw.mean()) / (far_raw.max() - far_raw.min())   # 標準化避免高次方 overflow
idx = rng.permutation(len(far))
split = 8
train_idx, val_idx = idx[:split], idx[split:]

def make_features(v, degree=9):
    return np.hstack([v ** d for d in range(degree + 1)])

X_train, X_val = make_features(far[train_idx]), make_features(far[val_idx])
y_train, y_val = land_price[train_idx], land_price[val_idx]

w = np.zeros((X_train.shape[1], 1))
patience = 2000               # 容忍 2000 個 epoch 沒進步
best_val = np.inf             # 目前看過的最低驗證損失
best_w = w.copy()            # 最佳檢查點
best_epoch = 0
wait = 0                     # 已連續幾個 epoch 沒進步
val_curve = []

for epoch in range(60000):
    error = X_train @ w - y_train
    w -= 0.5 * (X_train.T @ error / len(X_train))
    val_loss = np.mean((X_val @ w - y_val) ** 2)
    val_curve.append(val_loss)
    if val_loss < best_val - 1e-6:   # 注意：要有實質進步才算數，避免被微小浮動誤導
        best_val, best_w, best_epoch = val_loss, w.copy(), epoch   # 創新低就存檔
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            break             # 連續沒進步達 patience，提早停止

w = best_w                   # 還原到最佳檢查點
print("實際跑到 epoch:", epoch, "（總上限 60000）")
print("最佳檢查點在 epoch:", best_epoch, "  最低驗證損失:", round(best_val, 4))

plt.figure(figsize=(6, 4))
plt.plot(val_curve, label="驗證損失")
plt.axvline(best_epoch, color="red", linestyle="--", label="最佳檢查點")
plt.xlabel("epoch")
plt.ylabel("驗證損失 MSE")
plt.title("提早停止：保留驗證損失最低的版本")
plt.legend()
plt.show()

## 學習曲線判讀與訓練診斷

學習曲線（learning curve）是把訓練損失與驗證損失隨 epoch 畫出來的圖，可以當成模型的體檢報告。收斂（convergence）指損失趨於平緩、不再大幅下降。

看圖決定動哪個旋鈕：
- 兩條都高且貼在一起：高偏差（high bias），也就是欠擬合，對策是加大模型、訓練更久或加更好的特徵。
- 訓練低、驗證高且落差大：高方差（high variance），也就是過擬合，對策是加正則化（dropout、提早停止）或拿更多資料。
- 兩條都低且收斂：剛好，維持即可。

原則是先看圖診斷，再決定下一步，不要盲目調參。

In [ ]:
# 概念：準備三組預造的損失曲線（欠擬合/剛好/過擬合），逐張對照判讀並印出對策
epochs = np.arange(1, 61)

# 造三種典型曲線當示範（用簡單函數仿真，不是真的訓練出來的）
underfit = {
    "train": 1.5 - 0.3 * (1 - np.exp(-epochs / 20)),   # 訓練損失也降不下去（停在高處）
    "val":   1.6 - 0.3 * (1 - np.exp(-epochs / 20)),
}
good_fit = {
    "train": 1.5 * np.exp(-epochs / 12) + 0.1,         # 兩條都降到低位且貼近
    "val":   1.5 * np.exp(-epochs / 12) + 0.15,
}
overfit = {
    "train": 1.5 * np.exp(-epochs / 8) + 0.05,         # 訓練持續降
    "val":   1.5 * np.exp(-epochs / 8) + 0.05 + 0.02 * epochs,   # 驗證反轉上升
}

def diagnose(train_curve, val_curve):
    final_train, final_val = train_curve[-1], val_curve[-1]
    gap = final_val - final_train
    if final_train > 0.8:                      # 連訓練都學不好
        return "高偏差（欠擬合）", "加大模型 / 訓練更久 / 加更好的特徵"
    if gap > 0.3:                              # 訓練好但驗證差很多
        return "高方差（過擬合）", "加正則化（dropout / early stopping）或拿更多資料"
    return "剛好（收斂良好）", "維持現狀即可"

cases = [("欠擬合", underfit), ("剛好", good_fit), ("過擬合", overfit)]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, curves) in zip(axes, cases):
    ax.plot(epochs, curves["train"], label="訓練")
    ax.plot(epochs, curves["val"], label="驗證")
    ax.set_title(name)
    ax.set_xlabel("epoch")
    ax.set_ylabel("損失")
    ax.legend()
plt.tight_layout()
plt.show()

for name, curves in cases:
    label, advice = diagnose(curves["train"], curves["val"])
    print(f"{name} -> 判讀: {label}; 對策: {advice}")

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）為樓高迴歸挑一個優化器（整合：小批次 + SGD/Adam）
#   情境：用自造的「樓層數對樓高」資料做一個最小迴歸，想知道哪個優化器收斂比較快。
#   要求：
#     1. 用 numpy 自造資料（樓層數對應樓高，約略線性 + 一點雜訊），切成固定 batch size 的小批次。
#     2. 同樣的網路分別用 SGD 與 Adam 各訓練數個 epoch，記錄每個 epoch 的訓練損失。
#     3. 把兩條損失曲線畫在同一張圖上。
#   驗收：應該看到兩條下降曲線，Adam 通常更早趨於平緩。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）抓出過擬合並用正則化救回（整合：驗證切分 + 過擬合 + dropout/正則化 + 學習曲線）
#   情境：用自造的「容積率與面積對地價」資料訓練一個容易過擬合的小網路。
#   要求：
#     1. 做 train/validation split，先訓練到驗證損失明顯反轉上升。
#     2. 在訓練流程中加入正則化（dropout 遮罩或權重衰減皆可）後重新訓練。
#     3. 把加入前後的訓練與驗證損失曲線並排比較。
#   驗收：應該看到加入正則化後，驗證損失反轉的幅度變小、訓練與驗證的落差縮小。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）自製訓練診斷器（整合：early stopping + 學習曲線 + 優化器 + 對策決策）
#   情境：把都市分區自造資料的訓練流程包成一個可重複實驗的小工具，讓它能自動停下並給診斷建議。
#   要求：
#     1. 寫一個訓練迴圈，內建 early stopping（含 patience 與最佳檢查點還原）。
#     2. 跑兩種設定（例如不同學習率，或有無正則化），各自輸出學習曲線。
#     3. 依曲線形狀以簡單規則自動判斷屬於欠擬合或過擬合，並印出建議的下一步對策。
#   驗收：應看到工具在驗證損失停滯時自動停止，並針對每次實驗印出「高偏差／高方差」判讀與一句對策建議。
# TODO: 在這裡完成


## 小結

- 小批次（mini-batch）讓更新更頻繁、記憶體更省，梯度雜訊還有助於跳出壞點；DataLoader 是自動分批與洗牌的搬運工，一個 epoch 的更新次數約為 樣本數 / batch size。
- 優化器決定怎麼用梯度更新參數：SGD 用固定步幅、Adam 自適應每個方向的步幅且通常收斂更快更穩；學習率始終是最關鍵的旋鈕。
- 切出驗證集才看得到泛化能力；訓練損失降、驗證損失反轉上升就是過擬合的訊號，兩條都高則是欠擬合。
- 丟棄法（dropout）訓練時隨機關神經元、推論時關閉並縮放補償；它和權重衰減一樣屬於正則化，能縮小訓練與驗證的落差。
- 提早停止（early stopping）盯著驗證損失，連續 patience 個 epoch 沒進步就停，並還原最佳檢查點，省時又避免過擬合。
- 學習曲線是模型的體檢報告：先看圖判斷高偏差或高方差，再決定加大模型、加正則化或補資料，不要盲目調參。